# setup + distance mapping

- creates `res/paired_measurements.csv` from `res/manual_measurements.csv`.
- creates `res/dcm_metadata.csv` (pixel spacing, slice thickness, laterality, control flag).
- optionally applies **manual NIfTI corrections** (writes `*_MOD.nii`).
- runs `process_joint_space()` for every patient NIfTI and writes per-patient outputs to `output_case/` or `output_control/`.

**Requirements**: `opencv-python` (`cv2`), `nibabel`, `pydicom`, `scipy`, `tqdm`, `seaborn`.

In [ ]:
import os
import glob
from glob import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
import seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.2)

import nibabel as nib
from scipy.ndimage import rotate
from pydicom import dcmread

# Local module (file is provided alongside this notebook)
from distance_mapping import process_joint_space

# -------------------------
# Paths
# -------------------------
DATASET_DIR = 'dataset'
RES_DIR = 'res'

MANUAL_MEASUREMENTS_CSV = os.path.join(RES_DIR, 'manual_measurements.csv')
PAIRED_MEASUREMENTS_CSV = os.path.join(RES_DIR, 'paired_measurements.csv')

CORONAL_CT_PATHS_CSV = os.path.join(RES_DIR, 'coronal_ct_paths.csv')
DCM_METADATA_CSV = os.path.join(RES_DIR, 'dcm_metadata.csv')

OUTPUT_CASE_DIR = 'output_case'
OUTPUT_CONTROL_DIR = 'output_control'

os.makedirs(RES_DIR, exist_ok=True)
os.makedirs(OUTPUT_CASE_DIR, exist_ok=True)
os.makedirs(OUTPUT_CONTROL_DIR, exist_ok=True)

print('DATASET_DIR:', DATASET_DIR)
print('RES_DIR:', RES_DIR)


In [ ]:
# -------------------------
# Pair with manual measurements
# -------------------------

df_manual = pd.read_csv(
    MANUAL_MEASUREMENTS_CSV,
    dtype={
        'Anon_MRN': str,
        'Xray_SL_distance': float,
    },
)

rows = []
for _, row in tqdm(df_manual.iterrows(), total=len(df_manual), desc='Processing manual measurements'):
    pat_id = str(row['Anon_MRN']).zfill(8)
    if pd.isna(row['Xray_SL_distance']):
        continue
    rows.append({'patient_id': pat_id, 'sl_distance': float(row['Xray_SL_distance'])})

df_paired = pd.DataFrame(rows).sort_values('patient_id').reset_index(drop=True)
df_paired.to_csv(PAIRED_MEASUREMENTS_CSV, index=False)

print('Wrote:', PAIRED_MEASUREMENTS_CSV, '| n =', len(df_paired))
df_paired.head()


In [ ]:
# -------------------------
# Create metadata (laterality + pixel geometry)
# -------------------------

# Patient IDs laterality (manually screened)
ids_flip = [
    '00054595','00083756','00278249','00302075','00302077','00302078','00302081','00302082',
    '00302087','00302088','00302089','00302092','00302095','00238493','00328925','00328930','00328931'
]

# Patient IDs control (manually screened)
ids_control = ['00328924','00328925','00328927','00328930']

# Loop through all patient IDs in the dataset folder
nii_files = [f for f in os.listdir(DATASET_DIR) if f.endswith('.nii')]
patient_ids_dataset = sorted([os.path.splitext(f)[0] for f in nii_files])

# Read coronal_ct_paths.csv (used to locate a DICOM file to read metadata)
df_paths = pd.read_csv(CORONAL_CT_PATHS_CSV)
if 'start' not in df_paths.columns:
    raise ValueError('coronal_ct_paths.csv must have a column named start')

# Optional fallback DICOM root, if the paths in coronal_ct_paths.csv are not local.
# Set the ALT_DICOM_ROOT environment variable to a directory of <patient_id>/*.dcm.
ALT_DICOM_ROOT = os.environ.get('ALT_DICOM_ROOT', '')

metadata = []
for patient_id in patient_ids_dataset:
    ds = None

    # 1) Try to find the path in coronal_ct_paths.csv
    for _, row in df_paths.iterrows():
        file_path = row['start']
        parts = os.path.normpath(file_path).split(os.sep)
        if len(parts) >= 4 and parts[3] == patient_id and os.path.exists(file_path):
            ds = dcmread(file_path)
            break

    # 2) Fallback directory: first DICOM file in ALT_DICOM_ROOT/patient_id/
    if ds is None and ALT_DICOM_ROOT and os.path.isdir(os.path.join(ALT_DICOM_ROOT, patient_id)):
        alt_dir = os.path.join(ALT_DICOM_ROOT, patient_id)
        dcm_files = [f for f in os.listdir(alt_dir) if f.lower().endswith('.dcm')]
        if dcm_files:
            ds = dcmread(os.path.join(alt_dir, dcm_files[0]))

    if ds is None:
        print(f'Patient ID {patient_id} not found in either path; skipping metadata row.')
        continue

    pixel_spacing = float(ds.PixelSpacing[0])
    slice_thickness = float(ds.SliceThickness)
    laterality = 'right' if patient_id in ids_flip else 'left'
    control = 1 if patient_id in ids_control else 0

    metadata.append({
        'PatientID': patient_id,
        'Laterality': laterality,
        'PixelSpacing': pixel_spacing,
        'SliceThickness': slice_thickness,
        'Control': control,
    })

metadata_df = pd.DataFrame(metadata).sort_values('PatientID').reset_index(drop=True)
metadata_df.to_csv(DCM_METADATA_CSV, index=False)

print('Wrote:', DCM_METADATA_CSV, '| n =', len(metadata_df))
metadata_df.head()


In [ ]:
# -------------------------
# Optional: manual NIfTI corrections (writes *_MOD.nii)
# -------------------------
# This consolidates the original ad-hoc commented fixes into a single, reproducible block.

def _load_unmod_or_base(patient_id: str) -> str:
    p_unmod = os.path.join(DATASET_DIR, f'{patient_id}_UNMOD.nii')
    p_base = os.path.join(DATASET_DIR, f'{patient_id}.nii')
    if os.path.exists(p_unmod):
        return p_unmod
    if os.path.exists(p_base):
        return p_base
    raise FileNotFoundError(f'No NIfTI found for {patient_id} (expected *_UNMOD.nii or .nii)')

# Fill/enable entries only if you truly want them applied.
# Spec keys supported: rotate_deg (+CCW), rotate_axes, flip_axis, transpose_axes
CORRECTIONS = {
    # '00083756': {'rotate_deg': 45, 'rotate_axes': (0, 1)},
    # '00302080': {'rotate_deg': 45, 'rotate_axes': (0, 1)},
    # '00302086': {'rotate_deg': 90, 'rotate_axes': (0, 1)},
    # '00302087': {'flip_axis': 1},
    # '00302089': {'rotate_deg': -45, 'rotate_axes': (0, 1)},
    # '00302093': {'transpose_axes': (1, 0, 2)},
}

def apply_corrections(corrections: dict):
    for pid, spec in corrections.items():
        in_path = _load_unmod_or_base(pid)
        out_path = os.path.join(DATASET_DIR, f'{pid}_MOD.nii')

        img = nib.load(in_path)
        data = img.get_fdata()

        if 'transpose_axes' in spec:
            data = np.transpose(data, spec['transpose_axes'])

        if 'flip_axis' in spec:
            data = np.flip(data, axis=int(spec['flip_axis']))

        if 'rotate_deg' in spec:
            deg = float(spec['rotate_deg'])
            axes = tuple(spec.get('rotate_axes', (0, 1)))
            data = rotate(data, angle=deg, axes=axes, reshape=True, order=1)

        nib.save(nib.Nifti1Image(data, np.eye(4)), out_path)
        print('Wrote:', out_path)

if CORRECTIONS:
    apply_corrections(CORRECTIONS)
else:
    print('CORRECTIONS is empty -> no *_MOD.nii files created.')


In [ ]:
# -------------------------
# Run distance mapping for all patients
# -------------------------

metadata_df = pd.read_csv(
    DCM_METADATA_CSV,
    dtype={'PatientID': str, 'PixelSpacing': float, 'Laterality': str, 'SliceThickness': float, 'Control': int}
)

def _pick_nifti_path(patient_id: str) -> str:
    p_mod = os.path.join(DATASET_DIR, f'{patient_id}_MOD.nii')
    p_base = os.path.join(DATASET_DIR, f'{patient_id}.nii')
    if os.path.exists(p_mod):
        return p_mod
    if os.path.exists(p_base):
        return p_base
    raise FileNotFoundError(f'Missing NIfTI for {patient_id}')

for _, row in metadata_df.iterrows():
    patient_id = row['PatientID']
    pixel_spacing = float(row['PixelSpacing'])
    slice_thickness = float(row['SliceThickness'])
    to_flip = str(row['Laterality']).lower() == 'right'
    is_control = int(row['Control']) == 1

    outdir = OUTPUT_CONTROL_DIR if is_control else OUTPUT_CASE_DIR
    nifti_path = _pick_nifti_path(patient_id)

    print(f'Processing {patient_id} | flip={to_flip} | outdir={outdir}')
    try:
        process_joint_space(
            nifti_path=nifti_path,
            pixel_spacing=pixel_spacing,
            slice_thickness=slice_thickness,
            patient_id=patient_id,
            outdir=outdir,
            flip=to_flip,
            save_fig=True,
        )
        print(f'Finished {patient_id}')
    except Exception as e:
        print(f'ERROR {patient_id}: {e}')
